In [17]:
import numpy as np
import torch
from torchvision import transforms
from torchvision import datasets
import torch.optim as optim
from torch.utils.data import DataLoader

In [18]:
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

train_dataset = datasets.MNIST(root='../dataset/mnist/',train=True,download=True,transform=transform)
test_dataset = datasets.MNIST(root='../dataset/mnist/',train=False,download=True,transform=transform)

train_load = DataLoader(train_dataset,shuffle=True,batch_size=BATCH_SIZE)
test_load = DataLoader(test_dataset,shuffle=False,batch_size=BATCH_SIZE)

In [19]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(784,512,bias=True)
        self.layer2 = torch.nn.Linear(512,256)
        self.layer3 = torch.nn.Linear(256,128)
        self.layer4 = torch.nn.Linear(128,64)
        self.layer5 = torch.nn.Linear(64,10)
        self.ReLu = torch.nn.ReLU()

    def forward(self,x):
        x =- x.view(-1,784)
        x = self.ReLu(self.layer1(x))
        x = self.ReLu(self.layer2(x))
        x = self.ReLu(self.layer3(x))
        x = self.ReLu(self.layer4(x))
        return self.layer5(x)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralNetwork().to(device)

In [20]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,momentum=0.5)

In [21]:
def train(epoch):
    running_loss = 0
    for batch_idx,data in enumerate(train_load,0):
        inputs,target = data
        inputs = inputs.to(device)
        target = target.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs,target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch , batch_idx + 1, running_loss / 300))
            running_loss = 0.0

In [26]:
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_load:
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _,predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print('Accuracy on test set: %d %%' % (100 * correct / total))

In [27]:
for epoch in range(1,11):
    train(epoch)
    test()

[1,   300] loss: 0.106
[1,   600] loss: 0.095
[1,   900] loss: 0.095
Accuracy on test set: 96 %
[2,   300] loss: 0.077
[2,   600] loss: 0.078
[2,   900] loss: 0.078
Accuracy on test set: 96 %
[3,   300] loss: 0.062
[3,   600] loss: 0.067
[3,   900] loss: 0.063
Accuracy on test set: 97 %
[4,   300] loss: 0.046
[4,   600] loss: 0.051
[4,   900] loss: 0.056
Accuracy on test set: 97 %
[5,   300] loss: 0.037
[5,   600] loss: 0.042
[5,   900] loss: 0.043
Accuracy on test set: 97 %
[6,   300] loss: 0.032
[6,   600] loss: 0.032
[6,   900] loss: 0.034
Accuracy on test set: 97 %
[7,   300] loss: 0.024
[7,   600] loss: 0.025
[7,   900] loss: 0.028
Accuracy on test set: 97 %
[8,   300] loss: 0.021
[8,   600] loss: 0.021
[8,   900] loss: 0.022
Accuracy on test set: 98 %
[9,   300] loss: 0.016
[9,   600] loss: 0.015
[9,   900] loss: 0.015
Accuracy on test set: 97 %
[10,   300] loss: 0.012
[10,   600] loss: 0.012
[10,   900] loss: 0.012
Accuracy on test set: 97 %
